---
title: "OMATA: floutage des visages"
author: "Anthony Hauser, Unisanté"
date: today
format:
  html:
    css: styles.css
    toc: true
    toc-location: left
    number-sections: true
    toc-depth: 4
    self-contained: true
    echo: false
page-layout: full
---

# Objectif

Anonymiser les images collectées en **floutant les visages**, entièrement **en local**
(aucune image n'est envoyée vers un service externe).

Deux détecteurs de visages sont comparés :

- **CenterFace** (modèle ONNX, exécuté via `onnxruntime`),
- **YuNet** (modèle ONNX, exécuté via le module `cv2.FaceDetectorYN` d'OpenCV).

Les fonctions réutilisables sont regroupées dans la bibliothèque `src/blur_utils.py`.
Les images floutées sont sauvegardées dans `results/blurred_images/<méthode>/`.

> Dépendances supplémentaires : `opencv-python`, `onnxruntime`, `numpy`.
> Installation : `pip install opencv-python onnxruntime numpy`

In [ ]:
#| include: false
# ===== Imports =====
from pathlib import Path
import importlib

import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Local module: add the project root to the path, then import the blur library.
import sys
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import src.blur_utils as bu
importlib.reload(bu)

print("Project root:", project_root)

# Modèles

Les deux modèles ONNX de détection de visage sont téléchargés une seule fois et
mis en cache dans `data_utils/models/` (dossier ignoré par git). Les exécutions
suivantes réutilisent les fichiers locaux.

| Modèle | Fichier | Taille approx. |
|--------|---------|----------------|
| CenterFace | `centerface.onnx` | ~7 Mo |
| YuNet | `face_detection_yunet_2023mar.onnx` | ~350 Ko |

In [ ]:
# Download / retrieve the pre-trained models (cached locally).
centerface_model = bu.get_centerface_model()
yunet_model      = bu.get_yunet_model()

print("CenterFace:", centerface_model)
print("YuNet     :", yunet_model)

# Données

Nous travaillons sur un **sous-échantillon** des contrôles positifs
(`data/image_positive_controls/.../Images_test`). Les images sélectionnées sont
listées explicitement ci-dessous (par nom de fichier, pour une sélection
reproductible).

In [ ]:
# Source folder (positive controls).
image_folder = project_root / "data/image_positive_controls/Images_test-20240424T090736Z-001/Images_test"

# Explicit sub-sample: image numbers 6, 8, 9, 11, 20, 22-30, 34-36.
# File names are listed explicitly because extensions are mixed (.jpg/.png/.PNG)
# and number 36 exists as both 36.jpg and 36.PNG -> we keep 36.jpg.
selected_files = [
    "6.jpg", "8.jpg", "9.jpg", "11.jpg", "20.jpg",
    "22.PNG", "23.PNG", "24.png", "25.png", "26.png",
    "27.png", "28.png", "29.png", "30.png",
    "34.jpg", "35.png", "36.jpg",
]
image_paths = [image_folder / name for name in selected_files]

# Sanity check: make sure every selected file exists.
missing = [p.name for p in image_paths if not p.exists()]
assert not missing, f"Missing files: {missing}"
print(f"{len(image_paths)} images sélectionnées :", [p.name for p in image_paths])

# Fonctions de floutage

Deux fonctions sont définies dans `src/blur_utils.py` :

- `blur_faces_centerface(image_path, ...)` — détection CenterFace + floutage,
- `blur_faces_yunet(image_path, ...)` — détection YuNet + floutage.

Les deux retournent `(image_floutée_BGR, boîtes)` et partagent le même helper de
floutage (flou gaussien avec masque elliptique). Les fonctions `blur_image_paths(...)`
et `blur_folder(...)` appliquent une méthode à un lot d'images et sauvegardent les
résultats.

## Comparaison sur une image

On compare les deux méthodes sur une image avant de traiter tout le lot.

In [ ]:
# Pick one image and compare both methods.
sample_path = image_paths[0]
print("Image de test:", sample_path.name)

# Original image (BGR -> RGB for matplotlib display).
original = cv2.cvtColor(cv2.imread(str(sample_path)), cv2.COLOR_BGR2RGB)

# Blur with both detectors.
img_cf, boxes_cf = bu.blur_faces_centerface(sample_path)
img_yn, boxes_yn = bu.blur_faces_yunet(sample_path)

img_cf = cv2.cvtColor(img_cf, cv2.COLOR_BGR2RGB)
img_yn = cv2.cvtColor(img_yn, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(
    axes,
    [original, img_cf, img_yn],
    ["Original", f"CenterFace ({len(boxes_cf)} visage(s))", f"YuNet ({len(boxes_yn)} visage(s))"],
):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
display(Markdown("**Figure 1** : Comparaison du floutage CenterFace vs YuNet."))

# Floutage par lots et sauvegarde

Les images floutées sont sauvegardées dans `results/blurred_images/<méthode>/`,
en conservant les noms de fichiers d'origine. Ce dossier de résultats est ignoré
par git (sorties régénérables).

In [ ]:
# Output folder and detectors to run.
output_root = project_root / "results/blurred_images"
methods = ["centerface", "yunet"]

# Blur the selected images with each method and save them.
summaries = {}
for method in methods:
    out_dir = output_root / method
    print(f"\n=== {method} ===")
    summaries[method] = bu.blur_image_paths(image_paths, out_dir, method=method)

## Résumé

Nombre de visages détectés par méthode.

In [ ]:
# Per-method summary: number of images and detected faces.
rows = []
for method, summary in summaries.items():
    n_images = len(summary)
    n_faces = sum(s["n_faces"] for s in summary)
    n_with_face = sum(1 for s in summary if s["n_faces"] > 0)
    rows.append({
        "method": method,
        "n_images": n_images,
        "n_images_with_face": n_with_face,
        "n_faces_total": n_faces,
    })

df_summary = pd.DataFrame(rows)
display(df_summary)
display(Markdown("**Tableau 1** : Nombre de visages détectés et floutés par méthode."))